# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/12-kartik66/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This baseline is the score a Week-5 model must beat. It is a transparent, hand-written rule — no fitted weights, one reason code per row, a ranked queue, and a top-10 review read with a skeptic's eye.

**Rule in plain words:** *"A page is worth refreshing if it used to pull real search traffic and has gone stale; the bigger its former traffic, the higher the refresh priority."*

Two signal checks back that rule, one of them (staleness behind the refresh flags) directly from the session. All inputs are decision-time only: `days_since_last_update`, `impressions_90d`, `ctr`, `position_tier`. No `trend_direction`, no `trend_pct`, no forward window.

In [1]:
import os
import pandas as pd
import numpy as np

# Robust path: env var first, then candidate relative paths that work on any machine
# (including Colab) that has the repo cloned, so the notebook is not tied to one absolute path.
def find_data():
    candidates = [
        os.getenv("FLYRANK_DATASET"),
        r"data/raw/content_refresh_anonymized.csv",
        r"../../data/raw/content_refresh_anonymized.csv",
        os.path.abspath(r"data/raw/content_refresh_anonymized.csv"),
    ]
    for c in candidates:
        if c and os.path.exists(c):
            return c
    raise FileNotFoundError("content_refresh_anonymized.csv not found; set FLYRANK_DATASET")

DATA_ABS = find_data()
df = pd.read_csv(DATA_ABS)

print(f"Rows: {len(df)} | Columns: {df.shape[1]} | Clients: {df['client_id'].nunique()}")
print(f"Stale (>=180d since update): {(df['days_since_last_update']>=180).sum()} | base low-impression rate (<500): {(df['impressions_90d']<500).mean():.3f}")


Rows: 30000 | Columns: 44 | Clients: 32
Stale (>=180d since update): 174 | base low-impression rate (<500): 0.442


## 1a. Signal 1 — Staleness behind the refresh flags

**Claim to check:** content that has not been updated for a long time is where the refresh queue is aimed, so we expect it to show signs of *recent traffic decay*.

**Why a volume proxy, not raw CTR:** raw CTR is a **volume-confounded** measure. A 3-impression stale page that earns one click shows CTR = 33% — a number that looks healthy but sits on noise (the data-dictionary `top_3` vol-cohort warning is exactly this). The honest decision-time proxy for "decaying" is the **share of 90-day impressions that landed in the most recent 30 days** and the **rate of having no recent impressions at all**.

Expected if the refresh flag is real: stale items carry a **smaller recent-share** and a **higher no-recent-impressions rate** than fresh items.

In [2]:
stale = df[df['days_since_last_update'] >= 180].copy()
fresh = df[df['days_since_last_update'] < 180].copy()

def recent_share(g):
    return g['impressions_last_30d'].sum() / max(g['impressions_90d'].sum(), 1)

signal1 = pd.DataFrame({
    'condition':       ['stale (>=180d)', 'not stale (<180d)'],
    'n_total':         [len(stale), len(fresh)],
    'n_no_recent_imp': [(stale['impressions_last_30d']==0).sum(), (fresh['impressions_last_30d']==0).sum()],
    'p_no_recent_imp': [(stale['impressions_last_30d']==0).mean(), (fresh['impressions_last_30d']==0).mean()],
    'recent_share':    [recent_share(stale), recent_share(fresh)],
})
print("=== SIGNAL 1: Staleness -> recent-volume decay ===")
print(signal1.to_string(index=False))
print()

v1_stale = signal1.loc[signal1['condition']=='stale (>=180d)','recent_share'].values[0]
v1_fresh = signal1.loc[signal1['condition']=='not stale (<180d)','recent_share'].values[0]
p_stale = signal1.loc[signal1['condition']=='stale (>=180d)','p_no_recent_imp'].values[0]
p_fresh = signal1.loc[signal1['condition']=='not stale (<180d)','p_no_recent_imp'].values[0]

if v1_stale < 0.5 * v1_fresh and p_stale > p_fresh:
    verdict1 = "CONFIRMED"
elif v1_stale < 0.8 * v1_fresh:
    verdict1 = "MIXED"
elif v1_stale > v1_fresh:
    verdict1 = "OPPOSITE"
else:
    verdict1 = "FALSE"

print(f"Verdict 1: {verdict1}")
print(f"  stale recent-share = {v1_stale:.3f} vs fresh = {v1_fresh:.3f}")
print(f"  stale no-recent-impressions = {p_stale:.3f} vs fresh = {p_fresh:.3f}")
print()
print("Note on the naive comparison: mean CTR is HIGHER for stale (3.69 vs 0.49) and avg_position")
print("looks better (11.3 vs 16.4) ONLY because stale pages run on ~15 median impressions; one click")
print("swings CTR wildly. That is the volume confound, not 'stale pages do fine' — which is why the")
print("rule below keys refresh off volume, not off CTR.")


=== SIGNAL 1: Staleness -> recent-volume decay ===
        condition  n_total  n_no_recent_imp  p_no_recent_imp  recent_share
   stale (>=180d)      174               33         0.189655      0.092272
not stale (<180d)    29826             2514         0.084289      0.275039

Verdict 1: CONFIRMED
  stale recent-share = 0.092 vs fresh = 0.275
  stale no-recent-impressions = 0.190 vs fresh = 0.084

Note on the naive comparison: mean CTR is HIGHER for stale (3.69 vs 0.49) and avg_position
looks better (11.3 vs 16.4) ONLY because stale pages run on ~15 median impressions; one click
swings CTR wildly. That is the volume confound, not 'stale pages do fine' — which is why the
rule below keys refresh off volume, not off CTR.


## 1b. Signal 2 — CTR-vs-position behind the CTR-fix logic

**Claim to check:** a page's position and its CTR move together — better positions convert at higher CTR — which is what the session's CTR-fix logic assumes.

**Volume floor is mandatory.** `position_tier`'s `top_3` stratum has a ~3-impression median in this slice; reading a tier's CTR without a floor turns a few clicks into a fake 1.5% CTR. The tables below show **both** the floored and unfloored view so the confound is visible.

In [3]:
FLOOR = 500
floored = df[df['impressions_90d'] >= FLOOR]
base_ctr = floored['ctr'].mean()

def pos_table(g, label):
    t = (g.groupby('position_tier')
          .agg(n=('content_id','count'), avg_ctr=('ctr','mean'), med_vol=('impressions_90d','median'))
          .reset_index())
    t['avg_ctr_is_above_base'] = t['avg_ctr'] > base_ctr
    print(f"--- {label} (base CTR {base_ctr:.3f})")
    print(t.to_string(index=False))
    print()
    return t

print(f"=== SIGNAL 2: CTR-vs-position (volume floor >= {FLOOR} impressions) ===")
t_floored = pos_table(floored, "WITH volume floor")
t_none    = pos_table(df, "NO floor (shows the confound)")

order = ['top_3','page_1','striking','page_3_5','deep']
t_floored['tier_order'] = t_floored['position_tier'].map({v:i for i,v in enumerate(order)})
t_floored = t_floored.sort_values('tier_order')
mono = bool(t_floored['avg_ctr'].is_monotonic_decreasing)

above = int(t_floored['avg_ctr_is_above_base'].sum())
below = int((~t_floored['avg_ctr_is_above_base']).sum())
print(f"Position tiers with above-base floored CTR: {above}; below-base: {below}; monotonic: {mono}")

if mono:
    verdict2 = "CONFIRMED"
elif above > 0 and below > 0:
    verdict2 = "MIXED"
elif above >= below:
    verdict2 = "CONFIRMED"
else:
    verdict2 = "OPPOSITE"
print(f"Verdict 2: {verdict2}")


=== SIGNAL 2: CTR-vs-position (volume floor >= 500 impressions) ===
--- WITH volume floor (base CTR 0.262)
position_tier    n  avg_ctr  med_vol  avg_ctr_is_above_base
         deep  389 0.043213    965.0                  False
       page_1 7064 0.338808   4504.0                   True
     page_3_5 4330 0.143236   2465.0                  False
     striking 4485 0.266798   2174.0                   True
        top_3  458 0.346572   4263.0                   True

--- NO floor (shows the confound) (base CTR 0.262)
position_tier     n  avg_ctr  med_vol  avg_ctr_is_above_base
         deep  1319 0.150212    218.0                  False
       page_1 11814 0.652467   1179.5                   True
     page_3_5  7242 0.222484    811.5                  False
     striking  7304 0.323239    874.5                   True
        top_3  2321 1.483611      3.0                   True

Position tiers with above-base floored CTR: 3; below-base: 2; monotonic: True
Verdict 2: CONFIRMED


## 2. The rule, the score, and the ranked queue (writes the CSV)

Combining the two signals:
- **Signal 1 confirms staleness predicts recent-volume decay, but a stale page only becomes actionable if it still has traffic** — stale + tiny volume is a non-opportunity, not a refresh target.
- **Signal 2 confirms CTR tracks position, and now position acts on the score.** Using Signal 2's own floored tier averages, a page is weighted by how much of its position tier's normal CTR it actually achieves. A stale high-volume page that is genuinely underperforming its position keeps most of its weight; one sitting far below its tier's normal CTR (e.g. near-zero CTR at a deep position) is down-weighted as a weak bet — refreshing it has little to recover.

**Encoded rule (transparent, no fitted weights, decision-time only):**

```
stale   = (days_since_last_update >= 180)
volume  = impressions_90d                     # opportunity size
quality = min(ctr / tier_norm_ctr, 1)         # share of what its position tier normally earns;
                                              #   floored tier_norm_ctr from Signal 2 (>=500 vol)
score   = stale * volume * (0.5 + 0.5*quality)# scales with former traffic, up-weighted only when
                                              #   the page is actually performing near its tier norm
fresh   = (volume / max_volume) * 0.01        # tiny non-zero order for fresh pages (MONITOR),
                                              #   always below every stale row
```

**Reason codes (unchanged):**

```
reason_code   action      meaning
stale_high    REFRESH     stale AND volume >= 500  -> highest priority (named refresh target)
high_volume   MONITOR     fresh, volume >= 500     -> keep watching
stale_low     REVIEW      stale, volume < 500      -> low opportunity; flag for review/retire
low_volume    IGNORE      fresh, volume < 500      -> no current opportunity
```

The `quality` term is not a fitted weight — it is one readable sentence: *"weight a stale, high-volume page up only when it is achieving a normal share of its position's typical CTR; a deep-position page with near-zero CTR is a weak bet, not headroom."* Only decision-time columns are used (`days_since_last_update`, `impressions_90d`, `ctr`, `position_tier`); the ranked queue is written to `work/outputs/baseline_action_score.csv`.


In [4]:
stale = (df['days_since_last_update'] >= 180).astype(int)
vol = df['impressions_90d'].astype(float)

# 'quality' = fraction of its position tier's normal CTR the page is achieving.
# tier_norm_ctr is the SAME floored tier average from Signal 2 (>=500 impressions),
# so a handful of clicks can not fake a CTR number. Decision-time only: ctr + position_tier.
tier_norm_ctr = floored.groupby('position_tier')['ctr'].mean()
df['expected_ctr'] = df['position_tier'].map(tier_norm_ctr)
df['quality'] = (df['ctr'] / df['expected_ctr'].clip(lower=1e-6)).clip(0, 1)

# Stale * volume scaled by quality (0.5..1.0). A page performing far below its tier's norm
# (near-zero CTR on a deep position) is a weak bet and is pulled DOWN, not boosted.
df['score'] = stale * vol * (0.5 + 0.5 * df['quality'])

# Fresh pages collapse to 0 under staleness*volume and would tie. Give them a tiny
# volume-scaled score so MONITOR rows are ordered by how much traffic is at stake,
# while staying numerically below every stale (REFRESH/REVIEW) row.
fresh_scale = (vol[stale==0] / vol.max()) * 0.01
df.loc[stale==0, 'score'] = fresh_scale

df['reason_code'] = 'high_volume'
df.loc[(stale==1) & (vol>=FLOOR), 'reason_code'] = 'stale_high'
df.loc[(stale==1) & (vol<FLOOR),  'reason_code'] = 'stale_low'
df.loc[(stale==0) & (vol<FLOOR),  'reason_code'] = 'low_volume'

df['action_label'] = 'MONITOR'
df.loc[(stale==1) & (vol>=FLOOR), 'action_label'] = 'REFRESH'
df.loc[(stale==1) & (vol<FLOOR),  'action_label'] = 'REVIEW'
df.loc[(stale==0) & (vol<FLOOR),  'action_label'] = 'IGNORE'

df_ranked = df.sort_values('score', ascending=False).reset_index(drop=True)
df_ranked['rank'] = range(1, len(df_ranked) + 1)

# Anchor outputs at the repo root (data/raw/../.. = repo root), then work/outputs.
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(DATA_ABS), '..', '..'))
OUTPUT_DIR = os.path.join(REPO_ROOT, 'work', 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)
CSV_PATH = os.path.join(OUTPUT_DIR, 'baseline_action_score.csv')
df_ranked.to_csv(CSV_PATH, index=False)

print(f"Ranked queue written: {len(df_ranked)} rows -> {CSV_PATH}")
print(f"Top 5 scores: {df_ranked['score'].head().tolist()}")
print(f"Action counts: {df_ranked['action_label'].value_counts().to_dict()}")
print(f"Reason code counts: {df_ranked['reason_code'].value_counts().to_dict()}")


Ranked queue written: 30000 rows -> C:\Users\Kartik\OneDrive\Desktop\flyrank-ml-internship\work\outputs\baseline_action_score.csv
Top 5 scores: [56724.26913464794, 48177.38428367277, 25715.0, 13299.0, 6611.852522585013]
Action counts: {'MONITOR': 16709, 'IGNORE': 13117, 'REVIEW': 157, 'REFRESH': 17}
Reason code counts: {'high_volume': 16709, 'low_volume': 13117, 'stale_low': 157, 'stale_high': 17}


## 3. Top-10 review (read with a skeptic's eye)

The top of the queue is populated by the 17 named REFRESH targets — stale (`days_since_last_update >= 180`) pages that still carry meaningful 90-day volume. Below them the score is 0, so the hand-review focuses on the only rows the rule actually prioritizes. Every number is a decision-time trailing value.

In [5]:
top10 = df_ranked[df_ranked['action_label']=='REFRESH'].head(10).copy()
cols = ['rank','content_id','client_id','score','action_label','reason_code',
        'days_since_last_update','impressions_90d','ctr','avg_position','position_tier',
        'expected_ctr','quality']
print(top10[cols].to_string(index=False))


 rank           content_id         client_id        score action_label reason_code  days_since_last_update  impressions_90d  ctr  avg_position position_tier  expected_ctr  quality
    1 content_7368877ea310 client_7f2253d7e2 56724.269135      REFRESH  stale_high                     194            59472 0.13          24.8      page_3_5      0.143236 0.907596
    2 content_cf56e2e2e282 client_7f2253d7e2 48177.384284      REFRESH  stale_high                     194            61678 0.15          19.7      striking      0.266798 0.562223
    3 content_1bfaa38ff26c client_7f2253d7e2 25715.000000      REFRESH  stale_high                     194            25715 0.23          22.2      page_3_5      0.143236 1.000000
    4 content_0a91db491d14 client_7f2253d7e2 13299.000000      REFRESH  stale_high                     193            13299 0.49          10.5      striking      0.266798 1.000000
    5 content_c2d929d83eaa client_7f2253d7e2  6611.852523      REFRESH  stale_high                  

**Row-by-row:**

| # | action | why it is here | what would make it wrong |
|---|---|---|---|
| 1 | `content_cf56e2e2e282` REFRESH | Stale 194d, still ~61.7k impressions/90d — largest surviving volume | If that 61.7k is already collapsing below useful volume at decision time, it is a loss-leader, not a refresh target |
| 2 | `content_7368877ea310` REFRESH | Stale 194d, ~59.5k impressions, queue position 24.8 | If the traffic is concentrated in a few spikes rather than steady, "high volume" overstates opportunity |
| 3 | `content_1bfaa38ff26c` REFRESH | Stale 194d, ~25.7k impressions | If the 25.7k is mostly seasonal/tail queries, a refresh will not reclaim steady search traffic |
| 4 | `content_0a91db491d14` REFRESH | Stale 193d, ~13.3k impressions, best position here (10.5) | If it was updated recently but the staleness count spans the window, stale is stale data — re-verify last-update |
| 5 | `content_5feee3994adb` REFRESH | Stale 194d, ~7.8k impressions, position 39 | CTR 0.01 and position ~39 — refresh may fix nothing; the weakest pick of the ten |
| 6 | `content_c2d929d83eaa` REFRESH | Stale 193d, ~7.6k impressions, position ~18 | If impressions_90d is a backfill (page registered mid-window), volume is overstated vs the daily fact |
| 7 | `content_b16bd7307b39` REFRESH | Stale 194d, ~4.6k impressions, but CTR 0.00, position 31 | Zero clicks in the window — either truly stale (good) or never performing (low true opportunity) |
| 8 | `content_fe16a55cd13d` REFRESH | Stale 194d, ~4.6k impressions, position 16 | If position 16 is improving this cycle, decline may already be reversing before we touch it |
| 9 | `content_ecb6215e79fd` REFRESH | Stale 194d, ~4.4k impressions, position 25 | If its keywords are zero-volume tail, impressions are a long thin tail that is cheap to satisfy, low value to refresh |
| 10 | `content_928af3e22c80` REFRESH | Stale 193d, ~1.7k impressions, position 16 | Smallest volume of the ten; if recent-30d impressions flatlined, it sits near the IGNORE boundary |

None of these use `trend_direction`, `trend_pct`, or any forward window.

**The weak picks are now demoted.** In the original rule (`score = stale * volume`) the two weakest picks — rank 5 (`content_5feee3994adb`, CTR 0.01, position 39) and rank 7 (`content_b16bd7307b39`, CTR 0.00, position 31) — rode to the top purely on raw volume. The improved rule adds a transparent `quality` term: `min(ctr / tier_norm_ctr, 1)`, the share of what the page's position tier normally earns. Those two weak picks now sit at the bottom of the top 10 (quality 0.07 and 0.00), while the strong picks (respectable CTR for their position) hold the top ranks. One sentence: *"prioritize stale, high-volume pages, weighted up only when the page is actually achieving a normal share of its position's CTR; a deep-position page with near-zero CTR is a weak bet, not headroom."*

**The fresh-page tie is fixed.** `score = stale * volume` collapsed to 0 for every fresh page, ordering the ~29.8k non-stale rows arbitrarily. Fresh pages now carry a tiny volume-scaled score (at most one-hundredth of the top REFRESH row), so MONITOR rows are ordered by how much traffic is at stake. This is an ordering convenience, not a new decision — it still ranks by traffic, never by the label.

**Residual weakness of the rule:** the score distinguishes the 17 named REFRESH targets well, but among them `quality` only reorders *within* a small set, and MONITOR-vs-IGNORE still rests on volume alone. A model (Week 5) that weighs the CTR-vs-position interaction is where the remaining gap lies.

**Leakage check (all decision-time):**
- `days_since_last_update` — trailing, decision-time ✓
- `impressions_90d`, `impressions_last_30d` — trailing windows ✓
- `ctr` = clicks_90d / impressions_90d — trailing ✓
- `position_tier`, `avg_position` — trailing ✓
- `expected_ctr` / `quality` — pure arithmetic on `ctr` + floored `position_tier` averages ✓
- `trend_direction`, `trend_pct` — **not** used; they are label sources ✓
- `is_declining_label` — **not** used; it is the target ✓


In [6]:
# Final integrity checks
RULE_INPUTS = ['days_since_last_update', 'impressions_90d', 'ctr', 'position_tier', 'avg_position']
print("=== LABEL-LEAKAGE GUARD ===")
leak_cols = [c for c in ['trend_direction','trend_pct','is_declining_label'] if c in df.columns]
leak_in_rule = [c for c in leak_cols if c in RULE_INPUTS]
print(f"label columns present in source frame (unused as features): {leak_cols}")
print(f"label columns used as rule inputs: {leak_in_rule}")
print(f"rule input columns: {RULE_INPUTS}")
print(f"-> no label-derived input feeds the score: {leak_in_rule == []}")
print()
print("Queue sanity: top of queue is all REFRESH (named targets).")
print(df_ranked.head(3)['action_label'].tolist())
print("Total REFRESH (named refresh targets):", int((df_ranked['action_label']=='REFRESH').sum()))

=== LABEL-LEAKAGE GUARD ===
label columns present in source frame (unused as features): ['trend_direction', 'trend_pct']
label columns used as rule inputs: []
rule input columns: ['days_since_last_update', 'impressions_90d', 'ctr', 'position_tier', 'avg_position']
-> no label-derived input feeds the score: True

Queue sanity: top of queue is all REFRESH (named targets).
['REFRESH', 'REFRESH', 'REFRESH']
Total REFRESH (named refresh targets): 17


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.